In [1]:
import pandas as pd
import statsmodels.api as sm

In [2]:
def backward_elimination(y, X, significance_level=0.10):
    """
    Iteratively removes variables with p-values > significance_level.
    
    """
    X = sm.add_constant(X)  # add intercept
    model = sm.OLS(y, X).fit()
    
    while True:
        # Get max p-value
        p_values = model.pvalues
        max_pval = p_values.max()
        worst_var = p_values.idxmax()
        
        # Stop if all p-values are <= significance_level
        if max_pval <= significance_level:
            break
        
        # Do not drop the constant
        if worst_var == "const":
            break
        
        # Drop worst variable
        print(f"Dropping '{worst_var}' (p-value = {max_pval:.4f})")
        X = X.drop(columns=[worst_var])
        
        # Refit model
        model = sm.OLS(y, X).fit()
    
    return model, X

# Benin

### Uploading data

In [ ]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataFoodCPI/Benin_completes"
df = pd.read_excel(x+".xlsx")


### Linear Regression

In [4]:

# Define X and y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Soybean oil ($/mt)' (p-value = 0.9820)
Dropping 'FAO Sugar Index' (p-value = 0.7932)
Dropping 'Maize ($/mt)' (p-value = 0.7649)
Dropping 'FAO Meat' (p-value = 0.5610)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.4652)
Dropping 'Sorghum ($/mt)' (p-value = 0.4215)
Dropping 'Sugar, world ($/kg)' (p-value = 0.4306)
Dropping 'Beef ** ($/kg)' (p-value = 0.5442)
Dropping 'FAO Dairy Index' (p-value = 0.5037)
Dropping 'FAO Cereals Index' (p-value = 0.1356)
Dropping 'Rice, Viet Namese 5% ($/mt)' (p-value = 0.1798)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.472
Model:                            OLS   Adj. R-squared:                  0.437
Method:                 Least Squares   F-statistic:                     13.46
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           2.56e-28
Time:                        05:39:34   Log-Likelihood:        

In [5]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.711
Model:                            OLS   Adj. R-squared:                  0.678
Method:                 Least Squares   F-statistic:                     21.20
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.88e-53
Time:                        05:39:34   Log-Likelihood:                -749.81
No. Observations:                 289   AIC:                             1562.
Df Residuals:                     258   BIC:                             1675.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [6]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.9733)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.9706)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.8463)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.711
Model:                            OLS   Adj. R-squared:                  0.682
Method:                 Least Squares   F-statistic:                     23.82
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.69e-55
Time:                        05:39:35   Log-Likelihood:                -749.83
No. Observations:                 289   AIC:                             1556.
Df Residuals:                     261   BIC:                             1658.
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                              

In [7]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.512
Model:                            OLS   Adj. R-squared:                  0.457
Method:                 Least Squares   F-statistic:                     9.356
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.91e-26
Time:                        05:39:35   Log-Likelihood:                -825.83
No. Observations:                 289   AIC:                             1712.
Df Residuals:                     259   BIC:                             1822.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [8]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.9086)
Dropping 'lag_Maize ($/mt)' (p-value = 0.8364)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.7914)
Dropping 'lag_FAO Meat' (p-value = 0.5385)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.4136)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.3702)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.4225)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.3274)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.2913)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.2162)
Dropping 'lag_const' (p-value = 0.1399)

Final model summary:
                                 OLS Regression Results                                
Dep. Variable:               Food CPI   R-squared (uncentered):                   0.571
Model:                            OLS   Adj. R-squared (uncentered):              0.541
Method:                 Least Squares   F-statistic:                              18.94
Date:                Thu, 02 Oct 2025   Prob (F-stat

In [9]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.728
Model:                            OLS   Adj. R-squared:                  0.696
Method:                 Least Squares   F-statistic:                     22.97
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.53e-56
Time:                        05:39:35   Log-Likelihood:                -741.49
No. Observations:                 289   AIC:                             1545.
Df Residuals:                     258   BIC:                             1659.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [10]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Maize ($/mt)' (p-value = 0.9997)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.8536)
Dropping 'lag_const' (p-value = 0.8638)
Dropping 'lag_FAO Meat' (p-value = 0.8829)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.8414)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.8086)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.7787)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.7103)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.5469)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.5116)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.3805)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.4245)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.3355)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.3137)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.2353)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.3442)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.1580)
Dropping 'lag_EUR/USD' (p-value = 0.1754)

Final model summary:
                                

# Burkina Faso

### Uploding data 

In [11]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Burkina_completes"
df = pd.read_excel(x+".xlsx")  

### Linear regression  with all the variables

In [ ]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


### Only Y lagged

In [14]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     61.85
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          7.88e-103
Time:                        05:39:36   Log-Likelihood:                -747.52
No. Observations:                 299   AIC:                             1557.
Df Residuals:                     268   BIC:                             1672.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [15]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Sunflower oil ($/mt)' (p-value = 0.9804)
Dropping 'Lamb ** ($/kg)' (p-value = 0.9772)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.9457)
Dropping 'EUR/USD' (p-value = 0.6620)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.5451)
Dropping 'Sugar, world ($/kg)' (p-value = 0.5708)
Dropping 'Palm oil ($/mt)' (p-value = 0.6480)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.5857)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.5373)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.5102)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.8489)
Dropping 'Soybean oil ($/mt)' (p-value = 0.4651)
Dropping 'Chicken ** ($/kg)' (p-value = 0.5323)
Dropping 'Maize ($/mt)' (p-value = 0.3917)
Dropping 'Sorghum ($/mt)' (p-value = 0.4928)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.1599)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.1295)
Dropping 'FAO Dairy Index' (p-value = 0.1315)

Final model summary:
                            OLS Regression Results                            
Dep. V

### Only X laaged

In [16]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.635
Model:                            OLS   Adj. R-squared:                  0.596
Method:                 Least Squares   F-statistic:                     16.16
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.34e-43
Time:                        05:39:37   Log-Likelihood:                -906.17
No. Observations:                 299   AIC:                             1872.
Df Residuals:                     269   BIC:                             1983.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [17]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_FAO Dairy Index' (p-value = 0.9717)
Dropping 'lag_FAO Oils Index' (p-value = 0.9800)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.9526)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.8363)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.7646)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.6180)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.5101)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.4578)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.4763)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.2902)
Dropping 'lag_Maize ($/mt)' (p-value = 0.2017)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.2915)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.2522)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.2315)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.624
Model:                            OLS   Adj. R-squared:                  0

### Y and X lagged

In [18]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


In [19]:

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.874
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     61.86
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          7.78e-103
Time:                        05:39:37   Log-Likelihood:                -747.51
No. Observations:                 299   AIC:                             1557.
Df Residuals:                     268   BIC:                             1672.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [20]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.9911)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.9741)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.9559)
Dropping 'lag_FAO Dairy Index' (p-value = 0.8985)
Dropping 'lag_FAO Sugar Index' (p-value = 0.8806)
Dropping 'lag_FAO Oils Index' (p-value = 0.6761)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.7191)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.6451)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.6083)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.4504)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.5551)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.3416)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.4248)
Dropping 'lag_const' (p-value = 0.4016)
Dropping 'lag_EUR/USD' (p-value = 0.2923)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.2713)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.2444)
Dropping 'lag_Maize ($/mt)' (p-value = 0.1784)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.2789)
Dropping '

We observe that the best model is the last one, Y_lagged and X_lagged the R_squared and R_squared adjusted are 89%

# Côte d'Ivoire

In [21]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Ivoire_completes"
df = pd.read_excel(x+".xlsx")

In [22]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Sugar, EU ($/kg)' (p-value = 0.9818)
Dropping 'Soybean oil ($/mt)' (p-value = 0.9246)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.9006)
Dropping 'FAO Sugar Index' (p-value = 0.8001)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.4840)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.3934)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.1937)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.2618)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.566
Model:                            OLS   Adj. R-squared:                  0.537
Method:                 Least Squares   F-statistic:                     19.62
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           3.28e-39
Time:                        05:39:38   Log-Likelihood:                -729.26
No. Observations:                 290   AIC:                             1497.
Df Residuals:      

In [23]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.811
Model:                            OLS   Adj. R-squared:                  0.792
Method:                 Least Squares   F-statistic:                     41.57
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           6.84e-79
Time:                        05:39:38   Log-Likelihood:                -606.51
No. Observations:                 289   AIC:                             1269.
Df Residuals:                     261   BIC:                             1372.
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [24]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Soybean oil ($/mt)' (p-value = 0.9996)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.6485)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.5109)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.4247)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.4099)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.5491)
Dropping 'Maize ($/mt)' (p-value = 0.3183)
Dropping 'Rice, Thai A.1 ($/mt)' (p-value = 0.3911)
Dropping 'Lamb ** ($/kg)' (p-value = 0.4373)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.4424)
Dropping 'Beef ** ($/kg)' (p-value = 0.3710)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.796
Method:                 Least Squares   F-statistic:                     71.09
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           3.20e-87
Time:                        05:39:38   

In [25]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.595
Model:                            OLS   Adj. R-squared:                  0.555
Method:                 Least Squares   F-statistic:                     14.83
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           6.76e-38
Time:                        05:39:38   Log-Likelihood:                -716.77
No. Observations:                 289   AIC:                             1488.
Df Residuals:                     262   BIC:                             1587.
Df Model:                          26                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [26]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_FAO Sugar Index' (p-value = 0.7317)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.6116)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.5015)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.4061)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.2623)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.1927)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.588
Model:                            OLS   Adj. R-squared:                  0.558
Method:                 Least Squares   F-statistic:                     19.16
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.01e-40
Time:                        05:39:38   Log-Likelihood:                -719.21
No. Observations:                 289   AIC:                             1480.
Df Residuals:                     268   BIC:                             1557.
Df Model:           

In [27]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.808
Model:                            OLS   Adj. R-squared:                  0.788
Method:                 Least Squares   F-statistic:                     40.61
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           7.62e-78
Time:                        05:39:38   Log-Likelihood:                -609.24
No. Observations:                 289   AIC:                             1274.
Df Residuals:                     261   BIC:                             1377.
Df Model:                          27                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [28]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.9703)
Dropping 'lag_FAO Sugar Index' (p-value = 0.8125)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.5115)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.4609)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.3358)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.3493)
Dropping 'lag_FAO Meat' (p-value = 0.3271)
Dropping 'lag_FAO Dairy Index' (p-value = 0.6809)
Dropping 'lag_FAO Cereals Index' (p-value = 0.5085)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.4977)
Dropping 'lag_Maize ($/mt)' (p-value = 0.2482)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.2558)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.2631)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.2144)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.2804)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.1706)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.3170)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.2037)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.1

# Guinée Bisseau

### Uploading data

In [29]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Guinee_completes"
df = pd.read_excel(x+".xlsx")


### Linear Regression

In [30]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'FAO Oils Index' (p-value = 0.9438)
Dropping 'Maize ($/mt)' (p-value = 0.8973)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.8623)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.8402)
Dropping 'Palm oil ($/mt)' (p-value = 0.7813)
Dropping 'Soybean oil ($/mt)' (p-value = 0.6908)
Dropping 'Rice, Viet Namese 5% ($/mt)' (p-value = 0.6272)
Dropping 'Rice, Thai 5%  ($/mt)' (p-value = 0.7342)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.5191)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3329)
Dropping 'Lamb ** ($/kg)' (p-value = 0.1347)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.620
Model:                            OLS   Adj. R-squared:                  0.590
Method:                 Least Squares   F-statistic:                     20.73
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.90e-38
Time:                        0

### Only Y lagged

In [31]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.862
Model:                            OLS   Adj. R-squared:                  0.842
Method:                 Least Squares   F-statistic:                     44.80
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           2.01e-76
Time:                        05:39:39   Log-Likelihood:                -490.58
No. Observations:                 247   AIC:                             1043.
Df Residuals:                     216   BIC:                             1152.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [32]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'FAO Oils Index' (p-value = 0.9661)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.9289)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.8730)
Dropping 'Palm oil ($/mt)' (p-value = 0.7048)
Dropping 'Maize ($/mt)' (p-value = 0.4560)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.5708)
Dropping 'Soybean oil ($/mt)' (p-value = 0.4478)
Dropping 'FAO Dairy Index' (p-value = 0.5767)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.2786)
Dropping 'Lamb ** ($/kg)' (p-value = 0.2856)
Dropping 'EUR/USD' (p-value = 0.2494)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.2346)
Dropping 'Rice, Viet Namese 5% ($/mt)' (p-value = 0.3078)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.2205)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.1303)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.1071)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.852
Model:                  

### Only X lagged

In [33]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.636
Model:                            OLS   Adj. R-squared:                  0.587
Method:                 Least Squares   F-statistic:                     13.06
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.86e-33
Time:                        05:39:39   Log-Likelihood:                -610.00
No. Observations:                 247   AIC:                             1280.
Df Residuals:                     217   BIC:                             1385.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [34]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.9714)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.9257)
Dropping 'lag_Maize ($/mt)' (p-value = 0.8684)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.8710)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.7792)
Dropping 'lag_FAO Oils Index' (p-value = 0.8344)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.3524)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.2359)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.2462)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.1250)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.1510)
Dropping 'lag_Rice, Thai 5%  ($/mt)' (p-value = 0.1111)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.1955)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.1690)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.612
Model:                            OLS   Adj. R-squared:       

In [35]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.848
Model:                            OLS   Adj. R-squared:                  0.827
Method:                 Least Squares   F-statistic:                     40.07
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           4.71e-72
Time:                        05:39:39   Log-Likelihood:                -502.34
No. Observations:                 247   AIC:                             1067.
Df Residuals:                     216   BIC:                             1175.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [36]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_FAO Oils Index' (p-value = 0.9394)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.9405)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.7523)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.6700)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.6728)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.5399)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.5299)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.5383)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.8261)
Dropping 'lag_FAO Sugar Index' (p-value = 0.5334)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.5306)
Dropping 'lag_const' (p-value = 0.6058)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.3502)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.3892)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.2719)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.1927)
Dropping 'lag_Rice, Thai 5%  ($/mt)' (p-value = 0.2746)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.3201)
Dropping 'lag_Maize ($/mt)' (p

# Mali

In [37]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Mali_completes"
df = pd.read_excel(x+".xlsx")

In [38]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.9438)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.8988)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.8762)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.6605)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.2961)
Dropping 'Soybean oil ($/mt)' (p-value = 0.2764)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.2567)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.492
Model:                            OLS   Adj. R-squared:                  0.452
Method:                 Least Squares   F-statistic:                     12.36
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           5.42e-30
Time:                        05:39:40   Log-Likelihood:                -883.23
No. Observations:                 304   AIC:                             1812.
Df Residuals:                     281   BIC:                   

In [39]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.873
Model:                            OLS   Adj. R-squared:                  0.859
Method:                 Least Squares   F-statistic:                     62.54
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          2.37e-104
Time:                        05:39:40   Log-Likelihood:                -670.27
No. Observations:                 303   AIC:                             1403.
Df Residuals:                     272   BIC:                             1518.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [40]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Soybean oil ($/mt)' (p-value = 0.9689)
Dropping 'Beef ** ($/kg)' (p-value = 0.8889)
Dropping 'Lamb ** ($/kg)' (p-value = 0.8714)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.8535)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.7792)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.4944)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.5578)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.873
Model:                            OLS   Adj. R-squared:                  0.862
Method:                 Least Squares   F-statistic:                     83.34
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          1.05e-110
Time:                        05:39:40   Log-Likelihood:                -670.81
No. Observations:                 303   AIC:                             1390.
Df Residuals:                     279   BIC:                            

In [41]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.510
Model:                            OLS   Adj. R-squared:                  0.458
Method:                 Least Squares   F-statistic:                     9.793
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           3.82e-28
Time:                        05:39:40   Log-Likelihood:                -875.33
No. Observations:                 303   AIC:                             1811.
Df Residuals:                     273   BIC:                             1922.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [42]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.8971)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.8341)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.7366)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.5054)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.4791)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.4090)
Dropping 'lag_const' (p-value = 0.2494)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.2031)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.2836)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.1946)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.1544)

Final model summary:
                                 OLS Regression Results                                
Dep. Variable:               Food CPI   R-squared (uncentered):                   0.607
Model:                            OLS   Adj. R-squared (uncentered):              0.580
Method:                 Least Squares   F-statistic:                              23.06
Date:                Thu, 02 Oct 2025 

In [43]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.871
Model:                            OLS   Adj. R-squared:                  0.856
Method:                 Least Squares   F-statistic:                     60.99
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          4.42e-103
Time:                        05:39:41   Log-Likelihood:                -673.58
No. Observations:                 303   AIC:                             1409.
Df Residuals:                     272   BIC:                             1524.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [44]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.9633)


Dropping 'lag_FAO Cereals Index' (p-value = 0.9638)
Dropping 'lag_Rice, Thai 5%  ($/mt)' (p-value = 0.8823)
Dropping 'lag_const' (p-value = 0.7600)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.6767)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.6221)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.5864)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.5996)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.5619)
Dropping 'lag_FAO Food Price Index' (p-value = 0.6476)
Dropping 'lag_FAO Dairy Index' (p-value = 0.7370)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.4810)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.4753)
Dropping 'lag_FAO Meat' (p-value = 0.4368)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.3683)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.3976)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.3277)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.2263)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.3389)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.

# Niger

In [45]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Niger_completes"
df = pd.read_excel(x+".xlsx")  


In [46]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'FAO Dairy Index' (p-value = 0.9577)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.8831)
Dropping 'Beef ** ($/kg)' (p-value = 0.8807)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.8105)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.6274)
Dropping 'Sugar, world ($/kg)' (p-value = 0.6017)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.6011)
Dropping 'Chicken ** ($/kg)' (p-value = 0.4429)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.2902)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.9115)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.2437)
Dropping 'Soybean oil ($/mt)' (p-value = 0.2447)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.538
Model:                            OLS   Adj. R-squared:                  0.509
Method:                 Least Squares   F-statistic:                     18.60
Date:                Thu, 02 Oct 2025   Prob (F-stat

In [47]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.883
Model:                            OLS   Adj. R-squared:                  0.869
Method:                 Least Squares   F-statistic:                     64.80
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          1.18e-102
Time:                        05:39:42   Log-Likelihood:                -664.26
No. Observations:                 289   AIC:                             1391.
Df Residuals:                     258   BIC:                             1504.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [48]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Lamb ** ($/kg)' (p-value = 0.9823)
Dropping 'Chicken ** ($/kg)' (p-value = 0.7348)
Dropping 'FAO Cereals Index' (p-value = 0.7235)
Dropping 'FAO Dairy Index' (p-value = 0.8121)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.7189)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.8224)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.6968)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.6096)
Dropping 'Soybean oil ($/mt)' (p-value = 0.5769)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.4576)
Dropping 'Beef ** ($/kg)' (p-value = 0.4351)
Dropping 'FAO Meat' (p-value = 0.5967)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.3886)
Dropping 'FAO Sugar Index' (p-value = 0.4098)
Dropping 'Sugar, world ($/kg)' (p-value = 0.6712)
Dropping 'FAO Food Price Index' (p-value = 0.2763)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.3622)
Dropping 'Rice, Viet Namese 5% ($/mt)' (p-value = 0.2790)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.3113)
Dropping 'Rice, Thai 5%  ($/mt)' (p-value = 0.264

In [49]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.568
Model:                            OLS   Adj. R-squared:                  0.520
Method:                 Least Squares   F-statistic:                     11.74
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           9.77e-33
Time:                        05:39:42   Log-Likelihood:                -852.83
No. Observations:                 289   AIC:                             1766.
Df Residuals:                     259   BIC:                             1876.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [50]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.9200)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.9051)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.8787)
Dropping 'lag_FAO Food Price Index' (p-value = 0.8345)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.8264)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.6542)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.4559)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.3816)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.3120)
Dropping 'lag_FAO Meat' (p-value = 0.4240)
Dropping 'lag_FAO Sugar Index' (p-value = 0.3825)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.3895)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.5077)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.5191)
Dropping 'lag_FAO Cereals Index' (p-value = 0.3387)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.1487)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-

In [51]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.884
Model:                            OLS   Adj. R-squared:                  0.870
Method:                 Least Squares   F-statistic:                     65.36
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          4.51e-103
Time:                        05:39:42   Log-Likelihood:                -663.17
No. Observations:                 289   AIC:                             1388.
Df Residuals:                     258   BIC:                             1502.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [52]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.9818)


Dropping 'lag_Beef ** ($/kg)' (p-value = 0.9516)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.9031)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.8490)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.8474)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.8275)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.8184)
Dropping 'lag_FAO Sugar Index' (p-value = 0.7807)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.6844)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.6675)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.6725)
Dropping 'lag_const' (p-value = 0.6978)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.5832)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.4241)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.3660)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.1419)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.2887)
Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.1244)
Dropping 'lag_FAO Cereals Index' (p-value = 0.1075)
Dropping 'lag_Sugar, world ($/kg)' (

# Sénégal

In [53]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Senegal_completes"
df = pd.read_excel(x+".xlsx")  


In [54]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Rice, Thai A.1 ($/mt)' (p-value = 0.9443)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.8423)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.7494)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.7757)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.5386)
Dropping 'Rice, Thai 5%  ($/mt)' (p-value = 0.5793)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.4845)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.3329)
Dropping 'Palm oil ($/mt)' (p-value = 0.2228)
Dropping 'Sorghum ($/mt)' (p-value = 0.1693)
Dropping 'FAO Sugar Index' (p-value = 0.1192)
Dropping 'Sugar, world ($/kg)' (p-value = 0.2546)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.619
Model:                            OLS   Adj. R-squared:                  0.597
Method:                 Least Squares   F-statistic:                     27.35
Date:                Thu, 02 Oct 2025   Pro

In [55]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.844
Model:                            OLS   Adj. R-squared:                  0.826
Method:                 Least Squares   F-statistic:                     48.91
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           4.33e-92
Time:                        05:39:43   Log-Likelihood:                -614.99
No. Observations:                 303   AIC:                             1292.
Df Residuals:                     272   BIC:                             1407.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [56]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.9822)
Dropping 'Palm oil ($/mt)' (p-value = 0.9482)
Dropping 'Sugar, world ($/kg)' (p-value = 0.9160)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.9060)
Dropping 'Soybean oil ($/mt)' (p-value = 0.8664)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.7564)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.7586)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.7343)
Dropping 'FAO Sugar Index' (p-value = 0.7003)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.7219)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.7582)
Dropping 'Rice, Thai 5%  ($/mt)' (p-value = 0.6446)
Dropping 'Beef ** ($/kg)' (p-value = 0.6464)
Dropping 'Lamb ** ($/kg)' (p-value = 0.5713)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.6062)
Dropping 'Rice, Viet Namese 5% ($/mt)' (p-value = 0.3641)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.3072)
Dropping 'FAO Cereals Index' (p-value = 0.3386)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.1300)
Dropping 'Wheat, US HRW ($/mt)'

In [57]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.649
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     17.42
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.42e-46
Time:                        05:39:44   Log-Likelihood:                -737.39
No. Observations:                 303   AIC:                             1535.
Df Residuals:                     273   BIC:                             1646.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [58]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.8065)
Dropping 'lag_FAO Oils Index' (p-value = 0.6984)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.3805)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.3394)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.2258)
Dropping 'lag_FAO Sugar Index' (p-value = 0.4300)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.2921)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.3062)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.2491)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.8353)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.2243)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.1803)
Dropping 'lag_FAO Cereals Index' (p-value = 0.3237)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.634
Model:                            OLS   Adj. R-squared:                  0.614
Method:                 

In [59]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.843
Model:                            OLS   Adj. R-squared:                  0.825
Method:                 Least Squares   F-statistic:                     48.54
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.01e-91
Time:                        05:39:44   Log-Likelihood:                -615.95
No. Observations:                 303   AIC:                             1294.
Df Residuals:                     272   BIC:                             1409.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [60]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_FAO Sugar Index' (p-value = 0.9394)
Dropping 'lag_FAO Cereals Index' (p-value = 0.8952)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.8481)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.7232)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.7415)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.6826)
Dropping 'lag_FAO Oils Index' (p-value = 0.6761)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.6540)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.5758)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.4750)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.3596)
Dropping 'lag_Palm oil ($/mt)' (p-value = 0.4121)
Dropping 'lag_FAO Meat' (p-value = 0.5176)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.5046)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.4294)
Dropping 'lag_Maize ($/mt)' (p-value = 0.3429)
Dropping 'lag_FAO Dairy Index' (p-value = 0.6439)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.1114)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.1

# Togo

In [61]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/Togo_completes"
df = pd.read_excel(x+".xlsx")


In [62]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month","Subsidy_Index","TradeRestriction_Index"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'FAO Oils Index' (p-value = 0.9560)
Dropping 'Maize ($/mt)' (p-value = 0.8028)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.7315)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.6383)
Dropping 'Sugar, world ($/kg)' (p-value = 0.5694)
Dropping 'Beef ** ($/kg)' (p-value = 0.5414)
Dropping 'Lamb ** ($/kg)' (p-value = 0.5258)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.4631)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.4355)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.1902)
Dropping 'Chicken ** ($/kg)' (p-value = 0.2122)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.1020)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.571
Model:                            OLS   Adj. R-squared:                  0.544
Method:                 Least Squares   F-statistic:                     21.28
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.53e

In [63]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.793
Model:                            OLS   Adj. R-squared:                  0.768
Method:                 Least Squares   F-statistic:                     32.86
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           2.65e-71
Time:                        05:39:45   Log-Likelihood:                -746.55
No. Observations:                 289   AIC:                             1555.
Df Residuals:                     258   BIC:                             1669.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [64]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Lamb ** ($/kg)' (p-value = 0.9946)
Dropping 'Chicken ** ($/kg)' (p-value = 0.9666)
Dropping 'Maize ($/mt)' (p-value = 0.9602)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.8909)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.7503)
Dropping 'FAO Dairy Index' (p-value = 0.6952)
Dropping 'FAO Oils Index' (p-value = 0.7094)
Dropping 'FAO Meat' (p-value = 0.7754)
Dropping 'FAO Cereals Index' (p-value = 0.5828)
Dropping 'FAO Food Price Index' (p-value = 0.4074)
Dropping 'Rice, Thai 25%  ($/mt)' (p-value = 0.5531)
Dropping 'Palm kernel oil ($/mt)' (p-value = 0.4265)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.4380)
Dropping 'FAO Sugar Index' (p-value = 0.3349)
Dropping 'Sugar, world ($/kg)' (p-value = 0.4792)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3204)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.4035)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.2799)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.3533)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.5192)
Dropping 'Rice

In [65]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.532
Model:                            OLS   Adj. R-squared:                  0.479
Method:                 Least Squares   F-statistic:                     10.13
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           1.43e-28
Time:                        05:39:45   Log-Likelihood:                -864.27
No. Observations:                 289   AIC:                             1789.
Df Residuals:                     259   BIC:                             1899.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [66]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.9816)
Dropping 'lag_FAO Dairy Index' (p-value = 0.9422)
Dropping 'lag_Maize ($/mt)' (p-value = 0.8852)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.8115)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.6370)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.5757)
Dropping 'lag_FAO Meat' (p-value = 0.4960)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.6086)
Dropping 'lag_Sugar, world ($/kg)' (p-value = 0.4310)
Dropping 'lag_FAO Sugar Index' (p-value = 0.3936)
Dropping 'lag_FAO Cereals Index' (p-value = 0.4746)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.6133)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.2128)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.2620)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.1890)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.1812)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:             

In [67]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.795
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     33.38
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           5.62e-72
Time:                        05:39:45   Log-Likelihood:                -744.76
No. Observations:                 289   AIC:                             1552.
Df Residuals:                     258   BIC:                             1665.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [68]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Maize ($/mt)' (p-value = 0.9082)
Dropping 'lag_const' (p-value = 0.8893)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.8108)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.7539)
Dropping 'lag_Chicken ** ($/kg)' (p-value = 0.7563)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.6870)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.6474)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.6723)
Dropping 'lag_FAO Sugar Index' (p-value = 0.6359)
Dropping 'lag_Rice, Thai 5%  ($/mt)' (p-value = 0.5421)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.7505)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.5084)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.5181)
Dropping 'lag_Sorghum ($/mt)' (p-value = 0.4371)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.5027)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.9123)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.3229)
Dropping 'lag_FAO Meat' (p-value = 0.3933)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)

# UEMOA

In [69]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/data/UEMOA_completes"
df = pd.read_excel(x+".xlsx")

In [70]:
# Définir X et y
X = df.drop(columns=["Food CPI", "Month"])   
X = sm.add_constant(X)       
y = df["Food CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Palm kernel oil ($/mt)' (p-value = 0.6640)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.5452)
Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.4295)
Dropping 'Sorghum ($/mt)' (p-value = 0.2758)
Dropping 'Maize ($/mt)' (p-value = 0.3518)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.3485)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.2682)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.3630)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.7189)
Dropping 'Soybean oil ($/mt)' (p-value = 0.1197)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.621
Model:                            OLS   Adj. R-squared:                  0.596
Method:                 Least Squares   F-statistic:                     24.51
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           5.41e-49
Time:                        05:39:45   Log-Likelihood:                

In [71]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.925
Model:                            OLS   Adj. R-squared:                  0.917
Method:                 Least Squares   F-statistic:                     111.8
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          6.64e-135
Time:                        05:39:46   Log-Likelihood:                -470.15
No. Observations:                 303   AIC:                             1002.
Df Residuals:                     272   BIC:                             1117.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
lag_y             

In [72]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'UCSB CHIRPS Rainfall: avg' (p-value = 0.7376)
Dropping 'Lamb ** ($/kg)' (p-value = 0.5475)
Dropping 'Sunflower oil ($/mt)' (p-value = 0.7035)
Dropping 'Sugar, EU ($/kg)' (p-value = 0.5674)
Dropping 'Wheat, US SRW ($/mt)' (p-value = 0.6214)
Dropping 'Soybean oil ($/mt)' (p-value = 0.5249)
Dropping 'Groundnut oil ** ($/mt)' (p-value = 0.5053)
Dropping 'Rice, Thai 5%  ($/mt)' (p-value = 0.4700)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.4847)
Dropping 'Wheat, US HRW ($/mt)' (p-value = 0.5154)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.924
Model:                            OLS   Adj. R-squared:                  0.919
Method:                 Least Squares   F-statistic:                     171.5
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          6.26e-145
Time:                        05:39:46   Log-Likelihood:                -

In [73]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()

print(model_2.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.652
Model:                            OLS   Adj. R-squared:                  0.615
Method:                 Least Squares   F-statistic:                     17.61
Date:                Thu, 02 Oct 2025   Prob (F-statistic):           5.69e-47
Time:                        05:39:46   Log-Likelihood:                -702.72
No. Observations:                 303   AIC:                             1465.
Df Residuals:                     273   BIC:                             1577.
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_const 

In [74]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.9377)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.5153)
Dropping 'lag_Wheat, US SRW ($/mt)' (p-value = 0.3334)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.3292)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.2613)
Dropping 'lag_FAO Cereals Index' (p-value = 0.1670)
Dropping 'lag_Maize ($/mt)' (p-value = 0.2047)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.2107)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.6224)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.5014)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.2110)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.638
Model:                            OLS   Adj. R-squared:                  0.615
Method:                 Least Squares   F-statistic:                     27.77
Date:                Thu, 02 Oct 2025   Prob (F-stat

In [75]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()

print(ols_model.summary())


                            OLS Regression Results                            
Dep. Variable:               Food CPI   R-squared:                       0.924
Model:                            OLS   Adj. R-squared:                  0.916
Method:                 Least Squares   F-statistic:                     110.6
Date:                Thu, 02 Oct 2025   Prob (F-statistic):          2.51e-134
Time:                        05:39:46   Log-Likelihood:                -471.65
No. Observations:                 303   AIC:                             1005.
Df Residuals:                     272   BIC:                             1120.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                      coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------------
lag_y     

In [76]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Lamb ** ($/kg)' (p-value = 0.9776)
Dropping 'lag_Rice, Thai 25%  ($/mt)' (p-value = 0.9820)
Dropping 'lag_FAO Cereals Index' (p-value = 0.9591)
Dropping 'lag_Wheat, US HRW ($/mt)' (p-value = 0.9269)
Dropping 'lag_Groundnut oil ** ($/mt)' (p-value = 0.8863)
Dropping 'lag_Sugar, EU ($/kg)' (p-value = 0.8992)
Dropping 'lag_FAO Meat' (p-value = 0.8235)
Dropping 'lag_Soybean oil ($/mt)' (p-value = 0.7997)
Dropping 'lag_Sunflower oil ($/mt)' (p-value = 0.5699)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.6515)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.9261)
Dropping 'lag_UCSB CHIRPS Rainfall: avg' (p-value = 0.5792)
Dropping 'lag_Rice, Viet Namese 5% ($/mt)' (p-value = 0.5364)
Dropping 'lag_Palm kernel oil ($/mt)' (p-value = 0.5440)
Dropping 'lag_Rice, Thai A.1 ($/mt)' (p-value = 0.3339)
Dropping 'lag_FAO Dairy Index' (p-value = 0.3287)
Dropping 'lag_FAO Food Price Index' (p-value = 0.6722)
Dropping 'lag_Beef ** ($/kg)' (p-value = 0.3108)
Dropping 'lag_Chic